In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data/customer_transactions.csv')

display(df.head(5))

,date,store_nbr,lylty_card_nbr,txn_id,prod_nbr,prod_qty,tot_sales,clean_prod_name,net_weight,brand,lifestage,premium_customer
0,2018-10-17,1,1000,1,5,2,6.0,natural chip compny seasalt 175 g,175,Natural Chip Company,YOUNG SINGLES/COUPLES,Premium
1,2019-05-14,1,1307,348,66,3,6.3,ccs nacho cheese 175 g,175,CCs,MIDAGE SINGLES/COUPLES,Budget
2,2019-05-20,1,1343,383,61,2,2.9,smiths crinkle cut chips chicken 170 g,170,Smith's,MIDAGE SINGLES/COUPLES,Budget
3,2018-08-17,2,2373,974,69,5,15.0,smiths chip thinly s cream onion 175 g,175,Smith's,MIDAGE SINGLES/COUPLES,Budget
4,2018-08-18,2,2426,1038,108,3,13.8,kettle tortilla chpshny jlpno chili 150 g,150,Kettle,MIDAGE SINGLES/COUPLES,Budget


In [2]:
brand_sales = df.groupby("brand")["tot_sales"].sum()

top_brands = (
    brand_sales.nlargest(10)
    .reset_index(name="total_sales")
)
top_brands["sales_share_%"] = top_brands["total_sales"] / brand_sales.sum() * 100

# tidy up the format to show 2 decimal places and a percentage sign
top_brands["sales_share_%"] = top_brands["sales_share_%"].map(lambda v: f"{v:.2f}%")

print(top_brands)

           brand  total_sales sales_share_%
0         Kettle     390239.8        21.70%
1        Doritos     226329.9        12.59%
2        Smith's     210440.0        11.70%
3       Pringles     177655.5         9.88%
4      Infuzions      99047.6         5.51%
5          Thins      88852.5         4.94%
6  Red Rock Deli      87607.5         4.87%
7       Twisties      81522.1         4.53%
8       Tostitos      79789.6         4.44%
9           Cobs      70569.8         3.92%


**Kettle is the standount brand** <br>
- accounting for a fifth of all sales, the strongest performance out of 20 brands<br>
<br>
**Dorito's, Smith's and Pringles are reliable, accounting for approx. 10-13% each** <br>
<br>
- these are the most well-received brands across the entire population. this is a great starting point and would help isolate top brands <br>
- positioning items from these brands on featured sections, and near the checkout isle would take advantage on their widespread appeal. <br>

**Prepare all the target segments and assign each to a variable**

In [3]:
# Naming: segment_<tier>_<lifestage> = target, comparison_<tier>_<lifestage> = its reference group.
# Each target gets its own comparison group, so nothing gets overwritten.

# budget older families
segment_b_olderfamilies = df[
    (df["lifestage"] == "OLDER FAMILIES") & (df["premium_customer"] == "Budget")
]
comparison_b_olderfamilies = df[
    df["lifestage"].isin(["OLDER SINGLES/COUPLES", "RETIREES"])
]

# budget young families
segment_b_youngfamilies = df[
    (df["lifestage"] == "YOUNG FAMILIES") & (df["premium_customer"] == "Budget")
]
comparison_b_youngfamilies = df[
    df["lifestage"].isin(["OLDER FAMILIES"])   # removed YOUNG FAMILIES: it contained the target
]

# mainstream young singles/couples
segment_main_youngsc = df[
    (df["lifestage"] == "YOUNG SINGLES/COUPLES") & (df["premium_customer"] == "Mainstream")
]
comparison_main_youngsc = df[
    df["lifestage"].isin(["YOUNG FAMILIES"]) & (df["premium_customer"] == "Budget")
]

# older singles/couples
segment_older_sc = df[df["lifestage"] == "OLDER SINGLES/COUPLES"]
comparison_older_sc = df[
    df["lifestage"].isin(["OLDER FAMILIES", "RETIREES"])
]

# mainstream retirees
segment_main_retirees = df[
    (df["lifestage"] == "RETIREES") & (df["premium_customer"] == "Mainstream")
]
comparison_main_retirees = df[
    df["lifestage"].isin(["OLDER SINGLES/COUPLES", "OLDER FAMILIES"])
]

# Safety check: no segment may share rows with its comparison group
for seg, comp in [
    (segment_b_olderfamilies,  comparison_b_olderfamilies),
    (segment_b_youngfamilies,  comparison_b_youngfamilies),
    (segment_main_youngsc,     comparison_main_youngsc),
    (segment_older_sc,         comparison_older_sc),
    (segment_main_retirees,    comparison_main_retirees),
]:
    assert seg.index.intersection(comp.index).empty, "segment overlaps its comparison group"

In [4]:
def classify_affinity(score):
    """Tiers are mirrored on the ratio scale (1.20 <-> 1/1.20), so over- and
    under-indexing are held to the same standard."""
    if pd.isna(score):
        return "No Data"
    if score >= 1.20:        return "Very High"      # 20%+ above
    elif score >= 1.10:      return "High"           # 10-20% above
    elif score >= 1.05:      return "Moderate High"  # 5-10% above
    elif score >= 1 / 1.05:  return "Neutral"        # roughly 0.95 to 1.05
    elif score >= 1 / 1.10:  return "Moderate Low"   # roughly 0.91 to 0.95
    elif score >= 1 / 1.20:  return "Low"            # roughly 0.83 to 0.91
    else:                    return "Very Low"       # below roughly 0.83


def affinity_table(segment, other, item_col="brand", value_col="prod_qty", z=1.96):
    """Affinity per item with a significance flag.

    significant = True when the score differs from 1 by more than sampling
    noise would explain (approximate 95% test on the log ratio). Rows are
    treated as independent, so this slightly overstates confidence because
    units from the same customer are correlated.
    """
    a = segment.groupby(item_col)[value_col].sum()
    b = other.groupby(item_col)[value_col].sum()
    A, B = a.sum(), b.sum()

    t = pd.DataFrame({"seg_units": a, "other_units": b}).fillna(0)
    with np.errstate(divide="ignore", invalid="ignore"):
        t["affinity"] = (t["seg_units"] / A) / (t["other_units"] / B)
        se = np.sqrt(1 / t["seg_units"] - 1 / A + 1 / t["other_units"] - 1 / B)
        t["significant"] = np.abs(np.log(t["affinity"])) > z * se
    t["tier"] = t["affinity"].map(classify_affinity)
    return t.sort_values("affinity", ascending=False)

brand_affinity = affinity_table(segment_main_youngsc, comparison_main_youngsc, item_col="brand")

display(brand_affinity)

,seg_units,other_units,affinity,significant,tier
brand,,,,,
Twisties,1673,1202,1.321242,True,Very High
Cobs,1617,1185,1.295337,True,Very High
Kettle,7172,5328,1.277812,True,Very High
Pringles,4326,3261,1.259293,True,Very High
Tyrrells,1143,864,1.255809,True,Very High
Tostitos,1645,1266,1.233455,True,Very High
Doritos,4447,3442,1.226443,True,Very High
Infuzions,2343,1933,1.150619,True,High
Thins,2187,1846,1.124626,True,High


there is a significant difference in the brand mix of purchases made by Mainstream Young Singles/Couples vs Budget Young Families

the following brand affinity scores are computed comparing the preference of Mainstream Young Singles/Couples, and Budget Young Families <br>

these two are compared due to the following reasons: <br>

- similar sales and population % share
- both under the younger cohort umbrella
- different lifestage


*in general, mainstream Young/Singles couples only pick out chips from certain brands, resulting in lopsided affinity scores*

- 7 brands with very high resonance
- 2 highly sought after 
- 2 neutral and insignificant result; could be sales out of luck 
- 9 brands with very low pickup rate


*mainstream young s/c have narrow brand preferences, compared to Young Families counterpart* <br>
*this could also point towards an opposing preference in brands between singles/couples ana family lifestages*

In [5]:
# compare budget young families with BUDGET older families counterpart
segment_b_youngfamilies = df[
    (df["lifestage"] == "YOUNG FAMILIES") & (df["premium_customer"] == "Budget")
]
comparison_b_youngfamilies = df[
    df["lifestage"].isin(["OLDER FAMILIES"]) & (df["premium_customer"] == "Budget")
]



brand_affinity = affinity_table(segment_b_youngfamilies, comparison_b_youngfamilies, item_col="brand")

display(brand_affinity)

,seg_units,other_units,affinity,significant,tier
brand,,,,,
Tostitos,1266,1365,1.124910,True,High
Tyrrells,864,953,1.099607,True,Moderate High
French Fries,242,276,1.063464,False,Moderate High
CCs,768,882,1.056110,False,Moderate High
Doritos,3442,3953,1.056089,True,Moderate High
Sunbites,498,590,1.023750,False,Neutral
Pringles,3261,3865,1.023335,False,Neutral
Grain Waves,1091,1296,1.021025,False,Neutral
Infuzions,1933,2303,1.018016,False,Neutral


This means that, in general, Budget Young Families has a purchase mix similar to that of Budget Older Families <br>
- 5 brands [1:high+4:moderatehigh] being the exception, young families lean over these brands. 

How the score behaves

The score is a ratio of two shares, so each value means this:

*Near 1:* the brand takes about the same slice of both groups’ purchases, so the two groups buy it at similar rates. <br>
*Above 1 or below 1:* the segment leans toward or away from the brand, and the further from 1, the bigger the gap.

could the gap between these two groups’ brand shares plausibly come from random variation alone? 
- Scores near 1 with almost everything flagged False answers **“yes, for nearly every brand.”** 

**The two cohorts have a statistically indistinguishable brand mix.** The sales themselves are real and have causes, but the groups don’t differ in what they pick.

In [6]:
# compare budget young families with ALL older families
segment_b_youngfamilies = df[
    (df["lifestage"] == "YOUNG FAMILIES") & (df["premium_customer"] == "Budget")
]
comparison_b_youngfamilies = df[
    df["lifestage"].isin(["OLDER FAMILIES"])
]



brand_affinity = affinity_table(segment_b_youngfamilies, comparison_b_youngfamilies, item_col="brand")

display(brand_affinity)

,seg_units,other_units,affinity,significant,tier
brand,,,,,
French Fries,242,544,1.133853,False,High
CCs,768,1824,1.073190,False,Moderate High
Tostitos,1266,3011,1.071675,True,Moderate High
Doritos,3442,8206,1.069103,True,Moderate High
Sunbites,498,1206,1.052501,False,Moderate High
Tyrrells,864,2120,1.038767,False,Neutral
Cheezels,642,1593,1.027211,False,Neutral
Infuzions,1933,4846,1.016690,False,Neutral
Kettle,5328,13367,1.015946,False,Neutral


the similarities in brand mix remain evident despite the larger comparison cohort, being the Older Families with budget, mainstream, and premium tiers all combined. <br>
- top brands with high to moderate high slightly shuffle in places, but retaining the top 5 composition [1H+4mH]


In [7]:
# compare budget young families with ALL older families
# mainstream retirees
segment_main_retirees = df[
    (df["lifestage"] == "RETIREES") & (df["premium_customer"] == "Mainstream")
]
comparison_main_retirees = df[
    df["lifestage"].isin(["OLDER SINGLES/COUPLES", "OLDER FAMILIES"]) & (df["premium_customer"] == "Mainstream")
]

comparison_older_sc = df[
    df["lifestage"].isin(["OLDER SINGLES/COUPLES"])
]


brand_affinity = affinity_table(segment_main_retirees, comparison_older_sc, item_col="brand")

display(brand_affinity)

,seg_units,other_units,affinity,significant,tier
brand,,,,,
CCs,667,1612,1.066272,False,Moderate High
Burger Rings,228,553,1.062471,False,Moderate High
Grain Waves,1258,3064,1.058033,False,Moderate High
Sunbites,456,1111,1.057689,False,Moderate High
Twisties,1516,3767,1.037077,False,Neutral
Cheetos,441,1102,1.031251,False,Neutral
Thins,2279,5723,1.026190,False,Neutral
Cheezels,720,1816,1.021701,False,Neutral
Infuzions,2246,5665,1.021685,False,Neutral
